# Qwen3-1.7B tiny M1 capacity comparison

Compare the same 504 training rows and unchanged evaluation/gates against the preserved 0.6B run. Pin Qwen3-1.7B at revision 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e. Keep the same twelve entities, LoRA rank, batches, optimizer settings, 120 attempts, question splits and sampling policies. This is a model/checkpoint feasibility comparison, not a final model selection or proof that size alone causes differences.

First run a disposable GPU memory probe: one temporary optimizer step plus a representative batch-16 generation. It saves no adapter, reports no learning metrics, and exits its process. Only after it passes, start the actual trial from fresh pinned base weights/new adapter. The real trial measures its own M0 fact/utility baseline; it does not reuse 0.6B responses as expected 1.7B outputs. Preserve accidental knowledge matches rather than filtering data.

No training beyond the original 120-attempt budget. No automatic batch/precision/quantization fallback on OOM. Use the existing Kaggle GPU kernel/HF_TOKEN and publish the new source/config/public metadata/control report before running. Passing a tiny trial would still require a full 1.7B candidate audit/release and validated M1/M2/recovery later.


In [ ]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='qwen17-m1-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/training/model_capacity.py').is_file(), 'Publish the new 1.7B comparison source/config/metadata before running this notebook.'
assert Path('configs/training/m1_tiny_qwen17_v1.json').is_file()
assert Path('experiments/runs/m1_tiny_paraphrase_attempt_20261106/summary.json').is_file(), 'Publish the verified 0.6B result control too.'



In [ ]:
assert Path('/kaggle/working').is_dir()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)


In [ ]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
artifact_repo = 'tharundevulapalli/persistent-llm-unlearning-checkpoints'
api = HfApi(token=hf_token)
assert api.repo_info(artifact_repo, repo_type='model').private
print('Private artifact repository is accessible. Token is not displayed.')

comparison_config = json.loads(Path('configs/training/m1_tiny_qwen17_v1.json').read_text())
assert comparison_config['model_id'] == 'Qwen/Qwen3-1.7B'
assert comparison_config['model_revision'] == '70d244cc86ccca08cf5af4e1e306ecf908b1ad5e'
def run_trial_mode(mode, directory, check=True):
    child_env = os.environ.copy()
    child_env['HF_TOKEN'] = hf_token
    try:
        return subprocess.run([sys.executable, '-u', '-m', 'src.training.mixed_m1', '--config', 'configs/training/m1_tiny_qwen17_v1.json', '--output-dir', str(directory), '--mode', mode], env=child_env, check=check)
    finally:
        child_env.pop('HF_TOKEN', None)


In [ ]:
prep_dir = Path(tempfile.mkdtemp(prefix='qwen17-m1-prepare-', dir=remote_root))
run_trial_mode('prepare', prep_dir)
report = json.loads((prep_dir / 'masking_report.json').read_text())
audit = json.loads((prep_dir / 'augmentation_audit.json').read_text())
assert report['rows'] == 504 and report['prefixes_verified'] and report['truncated_examples'] == 0
for mode in ['answer_only', 'thinking']:
    assert report['by_mode'][mode]['rows'] == 252 and report['by_mode'][mode]['max_tokens'] <= 256
assert audit['new_templates'] == 14 and audit['unique_training_questions'] == 252
print('Augmentation audit:', json.dumps(audit, indent=2))
rows = [json.loads(line) for line in (prep_dir / 'selected_train.jsonl').read_text().splitlines()]
for row in [r for r in rows if not r['thinking_enabled'] and r['training_question_variant'] != 'original'][:4]:
    print('New training-only question:', row['question'])
example = next(r for r in rows if r['thinking_enabled'] and r['task_type'] == 'author_to_year' and r['training_question_variant'] != 'original')
print('Constructed target:', example['assistant_target'])
print('Original candidate/development/final wording and arithmetic probes remain unchanged.')

print('New model pin:', comparison_config['model_id'], comparison_config['model_revision'])
print('Baseline policy:', comparison_config['baseline_policy'])


In [ ]:
# Disposable fit process: one temporary update, no checkpoint and no learning score.
import zipfile

fit_dir = Path(tempfile.mkdtemp(prefix='qwen17-memory-fit-', dir=remote_root))
print('Memory probe outputs:', fit_dir, flush=True)
fit_result = run_trial_mode('fit', fit_dir, check=False)
fit_path = fit_dir / 'fit_report.json'
fit_archive = Path(str(fit_dir) + '-evidence.zip')
with zipfile.ZipFile(fit_archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(fit_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Memory evidence ZIP:', fit_archive)
assert fit_path.is_file(), 'Memory probe failed before a report; inspect traceback and preserve this ZIP.'
fit_report = json.loads(fit_path.read_text())
print(json.dumps(fit_report, indent=2))
assert fit_result.returncode == 0 and fit_report['success'], 'Stop: memory probe failed. Keep batching/precision fixed and report the evidence before changing the comparison.'
assert fit_report['model_id'] == comparison_config['model_id'] and fit_report['model_revision'] == comparison_config['model_revision']
print('Probe process exited. Its temporary adapter was discarded; actual training starts fresh in the next cell.')


In [ ]:
# Actual comparison: new process, fresh 1.7B base/new adapter, 120 attempts.
assert fit_report['success']
output_dir = Path(tempfile.mkdtemp(prefix='m1-tiny-qwen17-v1-', dir=remote_root))
print('Training outputs:', output_dir, flush=True)
result = run_trial_mode('train', output_dir, check=False)
from shutil import copyfile
copyfile(fit_dir / 'fit_report.json', output_dir / 'memory_fit_report.json')
copyfile(fit_dir / 'plan.json', output_dir / 'memory_fit_plan.json')
assert result.returncode in (0, 2), 'Trial failed; preserve partial outputs and inspect the traceback before retrying.'
if result.returncode == 2:
    print('The new model baseline utility gate failed: zero real-trial updates and no adapter. Preserve the evidence.')


In [ ]:
summary_path = output_dir / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('Execution complete:', summary['execution_complete'])
    print('Optimizer updates:', summary.get('optimizer_updates', 0))
    print('Pilot:', json.dumps(summary.get('pilot', summary.get('base_utility_gate')), indent=2))
    if 'evaluation' in summary:
        for stage, groups in summary['evaluation'].items():
            for name, group in groups.items():
                print(stage, name, '| answers:', str(group['answer_matches']) + '/' + str(group['count']), '| format:', group['format_violations'], '| caps:', group['token_limit_hits'], '| boundaries:', group['missing_thinking_boundaries'])
else:
    print('No complete summary. Inspect errors and preserve partial logs/checkpoint if available.')
print('This remains a tiny diagnostic. Preserve failed outcomes too.')

comparison_path = output_dir / 'model_comparison.json'
if comparison_path.is_file():
    comparison = json.loads(comparison_path.read_text())
    print('Model comparison (same recipe, one seed; not compute-matched):')
    for suite in ['dev_answer_only', 'dev_thinking', 'utility_thinking', 'train_probe_answer_only', 'train_probe_thinking']:
        old = comparison['control_post'][suite]
        candidate = comparison['candidate_post'][suite]
        print(suite, '| 0.6B:', str(old['answer_matches']) + '/' + str(old['count']), '| 1.7B:', str(candidate['answer_matches']) + '/' + str(candidate['count']))


In [ ]:
# Preserve adapter/optimizer/tokenizer and reports privately before the evidence export.
import hashlib
from src.m0_smoke_test import write_json

manifest_path = output_dir / 'artifact_manifest.json'
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    remote_path = 'diagnostics/m1-tiny-qwen17-v1/' + output_dir.name
    commit = api.upload_folder(repo_id=artifact_repo, repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve Qwen3-1.7B tiny M1 comparison')
    restore_dir = Path(tempfile.mkdtemp(prefix='qwen17-m1-restore-', dir=remote_root))
    for filename, expected in manifest['artifact_files_sha256'].items():
        restored = hf_hub_download(repo_id=artifact_repo, filename=remote_path + '/' + filename, revision=commit.oid, token=hf_token, local_dir=restore_dir)
        assert hashlib.sha256(Path(restored).read_bytes()).hexdigest() == expected, filename
    receipt = {'repo_id': artifact_repo, 'artifact_revision': commit.oid, 'path': remote_path, 'adapter_file_hashes_verified': True, 'canonical_m1': False}
    write_json(output_dir / 'persistence.json', receipt)
    api.upload_file(repo_id=artifact_repo, repo_type='model', path_or_fileobj=str(output_dir / 'persistence.json'), path_in_repo=remote_path + '/persistence.json', commit_message='Record verified Qwen3-1.7B adapter round-trip')
    print(json.dumps(receipt, indent=2))
else:
    print('No completed training adapter is available to upload. Preserve partial reports in the evidence ZIP.')


In [ ]:
import zipfile

archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Download from Kaggle Output:', archive)
print('Adapter/optimizer binaries stay in private Hugging Face storage. Preserve the receipt and ZIP before stopping the runtime.')
